# End-to-End Demo (Part 1): Market Selection -> Trade Download -> Cache

This notebook covers the first stage of the workflow:

1. Download market metadata and rank candidate questions.
2. Select one question using quantitative market-meta signals.
3. Download full historical trades for the selected market.
4. Cache trades + market context for downstream structural-break analysis.

The structural-break and news workflow is moved to:
- `examples/structural_break.ipynb`


## Step 0: Environment Setup

This cell ensures imports work whether the notebook is opened from the repo root or from `examples/`.


In [34]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "clients").exists() else cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/anuar.aimoldin/projects/polymarket_research


In [35]:
from __future__ import annotations

from datetime import UTC, datetime, timedelta
import json

import pandas as pd
from IPython.display import display

from clients.gamma_client import GammaClient
from collectors.markets_collector import MarketsCollector
from collectors.trades_collector import TradesCollector


## Step 1: Download Market Metadata and Build Candidate Set

We collect a broad metadata snapshot and use the built-in ranking (`market_score`) to shortlist candidates.

Notes:
- `top_n` controls shortlist size only. Universe download still fetches all matching markets.
- `min_created_at` helps reduce runtime by skipping older markets.
- `closed="false"` focuses on currently open questions.


In [3]:
gamma = GammaClient()
mc = MarketsCollector(gamma)

# Tune this cutoff to trade off coverage vs runtime.
meta_min_created_at = (datetime.now(tz=UTC) - timedelta(days=180)).replace(microsecond=0).isoformat().replace("+00:00", "Z")

report = mc.download_market_meta(
    include_active=True,
    include_inactive=False,
    closed="false",
    min_created_at=meta_min_created_at,
    limit=200,
    max_pages=None,
    top_n=30,
    show_progress=True,
    estimate_total=True,
)

markets_df = report["markets"]
summary = report["summary"]
top_markets_df = report["top_markets"]

print("Summary:")
summary


Summary:


{'generated_at_utc': '2026-02-17T12:54:51.668568+00:00',
 'markets_total': 27869,
 'markets_active': 27869,
 'markets_inactive': 0,
 'markets_closed': 0,
 'markets_archived': 0,
 'markets_open': 27869,
 'liquidity_total': 244083282.58483002,
 'liquidity_median': 1459.183,
 'volume_total': 1046032516.6237681,
 'volume_24h_total': 58427725.731582,
 'volume_1w_total': 326313782.70514715,
 'spread_median': 0.05}

In [4]:
display_cols = [
    c
    for c in [
        "condition_id",
        "question",
        "slug",
        "market_score",
        "volume_24h",
        "liquidity",
        "spread",
        "active",
        "closed",
        "end_date",
    ]
    if c in top_markets_df.columns
]

top_markets_df[display_cols].head(10)


,condition_id,question,slug,market_score,volume_24h,liquidity,spread,active,closed,end_date
0,0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13...,Will the Fed decrease interest rates by 50+ bp...,will-the-fed-decrease-interest-rates-by-50-bps...,0.998292,1.868453e+06,1.820371e+06,0.001,True,False,2026-03-18 00:00:00+00:00
1,0x25aa90b3cd98305e849189b4e8b770fc77fe89bccb7c...,Will the Fed increase interest rates by 25+ bp...,will-the-fed-increase-interest-rates-by-25-bps...,0.998226,9.382335e+05,1.705155e+06,0.001,True,False,2026-03-18 00:00:00+00:00
2,0x5e5c9dfaf695371a0cc321b47b35f66a6dbd1482f050...,"Will Bitcoin reach $150,000 in February?",will-bitcoin-reach-150k-in-february-2026,0.998123,6.208691e+05,1.925320e+06,0.001,True,False,2026-03-01 05:00:00+00:00
3,0x0b4cc3b739e1dfe5d73274740e7308b6fb389c5af040...,Will Jesus Christ return before 2027?,will-jesus-christ-return-before-2027,0.998084,8.787689e+05,1.601108e+06,0.001,True,False,2026-12-31 00:00:00+00:00
4,0x60c7d191ee2808417523e00ac85a4682324af4f1d887...,"Russia x Ukraine ceasefire by February 28, 2026?",russia-x-ukraine-ceasefire-by-february-28-2026,0.997747,6.288584e+05,4.470372e+05,0.001,True,False,2026-02-28 00:00:00+00:00
5,0xbc8d0415c38d2f8ac1ec5fa568015534999b79353aa2...,Will Elon Musk post 90-114 tweets from Februar...,elon-musk-of-tweets-february-14-february-16-90...,0.997461,6.391441e+05,3.240816e+06,0.001,True,False,2026-02-16 17:00:00+00:00
6,0x4c96c7e8460b8f9c4469483ddfa4d029c94605138dfc...,Will France win the Men's Ice Hockey gold meda...,will-france-win-the-mens-ice-hockey-gold-medal...,0.997149,5.857476e+05,2.219280e+05,0.001,True,False,2026-02-22 00:00:00+00:00
7,0x8be5e2483a67e35acdf7dd73473818b57cbe5b1d9a7f...,Will Elon Musk post 220-239 tweets from Februa...,elon-musk-of-tweets-february-10-february-17-22...,0.997013,5.616834e+05,2.530444e+05,0.001,True,False,2026-02-17 17:00:00+00:00
8,0xe1c67f75aac5b10dc28f1a2fbb79b079fc7f7320abfb...,"US strikes Iran by February 20, 2026?",us-strikes-iran-by-february-20-2026-151-468-62...,0.996923,7.964417e+05,1.776324e+05,0.001,True,False,2026-01-31 00:00:00+00:00
9,0x76047beb5fbb27a9ce3912c9fbb36c3c7eb5dc139c73...,Counter-Strike: FaZe vs Astralis (BO3) - PGL C...,cs2-faze-ast10-2026-02-17,0.996754,4.410821e+05,5.023180e+05,0.001,True,False,2026-02-17 16:00:00+00:00


## Step 2: Select the Target Question

By default, we pick the first row in `top_markets_df` (highest rank).

You can change `selected_idx` to inspect a different market from the shortlist.


In [5]:
selected_idx = 0
selected_market = top_markets_df.iloc[selected_idx]

condition_id = str(selected_market.get("condition_id"))
question = selected_market.get("question")
slug = selected_market.get("slug")

print("Selected market")
print(f"- index: {selected_idx}")
print(f"- condition_id: {condition_id}")
print(f"- slug: {slug}")
print(f"- question: {question}")


Selected market
- index: 0
- condition_id: 0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13f5915f050b615a193c20
- slug: will-the-fed-decrease-interest-rates-by-50-bps-after-the-march-2026-meeting
- question: Will the Fed decrease interest rates by 50+ bps after the March 2026 meeting?


## Step 3: Download Full Trade History for the Selected Market

We use `TradesCollector`, which downloads from the orderbook subgraph and normalizes to:

- `timestamp_utc`
- `price`
- `size`
- `outcome`
- `transaction_hash`

`estimate_total=True` attempts to provide a finite progress-bar total when supported by the active subgraph deployment.


In [6]:
tc = TradesCollector(gamma)

# Optional: set this to an ISO timestamp to limit the history window.
trade_start_date = None

df_trades = tc.download_all_trades(
    condition_id,
    start_date=trade_start_date,
    limit=500,
    max_pages=None,
    show_progress=True,
    estimate_total=True,
)

print(f"Downloaded trades: {len(df_trades)}")
df_trades.head()

Downloaded trades: 88428


,timestamp_utc,price,size,outcome,transaction_hash
0,2025-10-29 22:21:27+00:00,0.16,31250000.0,Yes,0xd91006abd45d7cbf97eb9a102a753bc4ea2064a1edfe...
1,2025-10-29 22:21:27+00:00,0.84,31250000.0,No,0xd91006abd45d7cbf97eb9a102a753bc4ea2064a1edfe...
2,2025-10-30 05:57:03+00:00,0.04,1041650.0,Yes,0xe6b19028d38176d452d2644ad9b794099d24f665c90d...
3,2025-10-30 05:57:03+00:00,0.96,1041650.0,No,0xe6b19028d38176d452d2644ad9b794099d24f665c90d...
4,2025-10-30 05:57:07+00:00,0.85,1000000.0,No,0x8a1c9648bdc6ee6fac6c7b9ff58290b1a3ec8a29c878...


## Step 4: Cache Trades for `structural_break.ipynb`

We persist the selected trades and market context so the structural-break notebook can run independently.


In [36]:
cache_dir = REPO_ROOT / "data" / "demo_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

trades_cache_path = cache_dir / "selected_trades.parquet"
context_cache_path = cache_dir / "selected_market_context.json"

# Save trades
pd.DataFrame(df_trades).to_parquet(trades_cache_path, index=False)

# Save lightweight context for downstream notebook
context_payload = {
    "condition_id": condition_id,
    "question": None if question is None else str(question),
    "slug": None if slug is None else str(slug),
    "selected_idx": int(selected_idx),
    "downloaded_at_utc": datetime.now(tz=UTC).isoformat(),
    "trade_rows": int(len(df_trades)),
    "trade_start_date": trade_start_date,
}
context_cache_path.write_text(json.dumps(context_payload, indent=2), encoding="utf-8")

print(f"Trades cached: {trades_cache_path}")
print(f"Context cached: {context_cache_path}")
context_payload


Trades cached: /Users/anuar.aimoldin/projects/polymarket_research/data/demo_cache/selected_trades.parquet
Context cached: /Users/anuar.aimoldin/projects/polymarket_research/data/demo_cache/selected_market_context.json


{'condition_id': '0xdeb615a52cd114e5aa27d8344ae506a72bea81f6ed13f5915f050b615a193c20',
 'question': 'Will the Fed decrease interest rates by 50+ bps after the March 2026 meeting?',
 'slug': 'will-the-fed-decrease-interest-rates-by-50-bps-after-the-march-2026-meeting',
 'selected_idx': 0,
 'downloaded_at_utc': '2026-02-18T09:26:34.775737+00:00',
 'trade_rows': 88428,
 'trade_start_date': None}

## Next Notebook

Open `examples/structural_break.ipynb` to continue with:
- EDA
- aggregation diagnostics
- structural break detection
- news pull around focal break
